# 02 - Labeling and Label Validation

This notebook is used to:

- extract frames
- manually label the frames
- check the labels
- validate the consistency of the keypoints before training

## Cell 1 — Imports and Config

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import deeplabcut
import tqdm

ROOT_NOTEBOOK = Path.cwd().resolve().parent
if str(ROOT_NOTEBOOK) not in sys.path:
    sys.path.append(str(ROOT_NOTEBOOK))

from src.paths import CONFIG_PATH, PROJECT_DIR, show_paths

show_paths()
assert CONFIG_PATH.exists(), f"config.yaml not found: {CONFIG_PATH}"

ROOT_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1
VIDEOS_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1\videos
RESULTS_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1\results
PROJECT_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1\bab_bar_only-Edo-2026-03-08
CONFIG_PATH: C:\Users\edoal\ProjetosPy\bab_dlc1\bab_bar_only-Edo-2026-03-08\config.yaml


## Cell 2 — Frame Extraction

### **Run this only if you have not extracted the frames yet**

In [ ]:
'''deeplabcut.extract_frames(
    str(CONFIG_PATH),
    mode="automatic",
    algo="uniform",
    userfeedback=False
)'''

## Cell 3 — Label Frames

In [ ]:
# Open the labeling interface
deeplabcut.label_frames(str(CONFIG_PATH))

## Cell 4 — Check Labels

In [ ]:
deeplabcut.check_labels(str(CONFIG_PATH))

Creating images with labels by Edo.


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:50<00:00,  1.02s/it]

If all the labels are ok, then use the function 'create_training_dataset' to create the training dataset!


## Cell 5 — Locate the Labels File

In [ ]:
from deeplabcut.utils import auxiliaryfunctions

cfg = auxiliaryfunctions.read_config(CONFIG_PATH)
project_path = Path(cfg["project_path"])
labeled_dir = project_path / "labeled-data"

h5_list = sorted(labeled_dir.rglob("CollectedData_*.h5"))
assert h5_list, f"No CollectedData_*.h5 found in {labeled_dir}"

labels_h5 = h5_list[0]
print("Labels file:", labels_h5)

df_lab = pd.read_hdf(labels_h5)
print("Shape:", df_lab.shape)
display(df_lab.head())

Arquivo de labels: C:\Users\edoal\ProjetosPy\bab_dlc1\bab_bar_only-Edo-2026-03-08\labeled-data\swept_sine_ready\CollectedData_Edo.h5
Shape: (50, 6)


scorer                                            Edo                          \
bodyparts                                   beam_left             beam_center   
coords                                              x           y           x   
labeled-data swept_sine_ready img0048.png  771.974683  342.849234  962.047918   
                              img0070.png  771.941775  342.911855  962.000874   
                              img0162.png  771.989905  343.022990  961.964484   
                              img0251.png  771.951897  342.948139  962.007539   
                              img0259.png  771.986719  343.090851  961.963167   

scorer                                                                          
bodyparts                                               beam_right              
coords                                              y            x           y  
labeled-data swept_sine_ready img0048.png  341.051834  1150.947401  336.977040  
                              img0070.png  341.012026  1150.880502  337.021975  
                              img0162.png  341.065382  1150.870963  336.968174  
                              img0251.png  341.029369  1150.907294  336.941623  
                              img0259.png  341.040511  1150.978803  336.986102

## Cell 6 — Check for NaN Values

In [ ]:
print("Total NaNs:", int(df_lab.isna().sum().sum()))
print("Frames with any NaN::", int(df_lab.isna().any(axis=1).sum()))

nan_rows = df_lab[df_lab.isna().any(axis=1)]
display(nan_rows.head())

Total NaNs: 0
Frames com qualquer NaN: 0


Empty DataFrame
Columns: [(Edo, beam_left, x), (Edo, beam_left, y), (Edo, beam_center, x), (Edo, beam_center, y), (Edo, beam_right, x), (Edo, beam_right, y)]
Index: []

## Cell 7 — Check for Flips

In [ ]:
scorer = df_lab.columns.get_level_values(0)[0]
parts = ["beam_left", "beam_center", "beam_right"]

X = np.vstack([df_lab[(scorer, p, "x")].to_numpy() for p in parts])
Y = np.vstack([df_lab[(scorer, p, "y")].to_numpy() for p in parts])

dx = X[2] - X[0]
flip = dx < 0

print("Frames with potential keypoint flip (dx<0):", int(np.nansum(flip)))

Frames com possível flip (dx<0): 0


## Cell 8 — Jitter Analysis

In [ ]:
def jitter(v):
    return np.sqrt(np.diff(v[0])**2 + np.diff(v[1])**2)

jit_left   = jitter((X[0], Y[0]))
jit_center = jitter((X[1], Y[1]))
jit_right  = jitter((X[2], Y[2]))

for name, j in [("left", jit_left), ("center", jit_center), ("right", jit_right)]:
    print(f"{name} jitter px: P50={np.nanpercentile(j,50):.2f}, P90={np.nanpercentile(j,90):.2f}, P99={np.nanpercentile(j,99):.2f}, Max={np.nanmax(j):.2f}")

left jitter px: P50=4.32, P90=29.57, P99=39.43, Max=41.88
center jitter px: P50=0.47, P90=2.63, P99=3.29, Max=3.50
right jitter px: P50=4.06, P90=29.36, P99=39.24, Max=41.85


## Cell 9 — Frames with Complete Data

In [ ]:
complete_mask = ~df_lab.isna().any(axis=1)
df_ok = df_lab[complete_mask]

print("Frames with complete data:", len(df_ok), "of", len(df_lab))

Frames completos: 50 de 50
